<a href="https://colab.research.google.com/github/MBAH05/QPS_Mimi_Project-2_EI/blob/main/Quantum-Phase_Preparation_mini_project_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MINI-PROJECT #1: QUANTUM STATE PREPARATION

Write a Qiskit function that takes a

\begin{equation}
2^n
\end{equation}

dimensional vector,

\begin{equation}
\phi \in \mathbb{C}^{2^n},
\end{equation}

such that

\begin{equation}
\| \psi \|_2 = 1,
\end{equation}

and outputs a circuit, such that,

\begin{equation}
U|0\rangle_n= \sum_{x=0}^{2^n - 1} \psi_x |x\rangle_n,
\end{equation}

The construction may use any number of ancillas, arbitrary 1-qubit gates and multi-controlled RZ gates, (the number of controlled may be arbitrarily large). No classical bit and measurements allowed.

Expectations:

• Documentation.

• Working Qiskit code for all n and demo for n = 4.

Note: This is a more open-ended problem, there are fairly straightforward, but inefficient ways to do (which are neverthelesss have great education value), but you can also try to aim for something more SOTA. For the very modern approach, I recommend this paper. I’m also happy to recommend other example

In [41]:
%pip install qiskit pylatexenc

In [42]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, AncillaRegister, ClassicalRegister
from qiskit.circuit.library import StatePreparation
from qiskit.quantum_info import Statevector, Operator, state_fidelity

import numpy as np
from typing import List, Dict, Any, Tuple

## Basic utilities

In [43]:
def normalize_state(vec: np.ndarray, tol: float = 1e-12) -> np.ndarray:
    vec = np.asarray(vec, dtype=complex).flatten()
    if vec.size == 0:
        raise ValueError("Empty state vector.")
    norm = np.linalg.norm(vec)
    if norm < tol:
        raise ValueError("Near-zero state vector.")
    return vec / norm


def num_qubits_from_state(vec: np.ndarray) -> int:
    N = len(vec)
    n = int(np.log2(N))
    if 2 ** n != N:
        raise ValueError(f"State dimension must be a power of 2, got {N}.")
    return n


def int_to_lsb_bits(x: int, m: int) -> List[int]:
    """Return little-endian bit list of length m."""
    return [(x >> i) & 1 for i in range(m)]


def hadamard_matrix(n: int) -> np.ndarray:
    H1 = np.array([[1, 1], [1, -1]], dtype=float) / np.sqrt(2)
    H = H1
    for _ in range(n - 1):
        H = np.kron(H, H1)
    return H


def sign_pm1_real(x: np.ndarray) -> np.ndarray:
    s = np.ones_like(np.real(x), dtype=int)
    s[np.real(x) < 0] = -1
    return s


def l2_error(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(np.asarray(a) - np.asarray(b)))

# David Gosset et al paper-inspired coarse states for real amplitudes

In [44]:
def build_coarse_bhbh_state_real(
    target_real: np.ndarray,
    num_random_trials: int = 64,
    seed: int = 1234
) -> Dict[str, Any]:
    """
    Heuristic coarse approximation of a real normalized state by
        |phi> = B2 H^n B1 H^n |0^n>.
    """
    psi = np.asarray(target_real, dtype=float).flatten()
    psi = psi / np.linalg.norm(psi)

    n = num_qubits_from_state(psi)
    N = len(psi)
    H = hadamard_matrix(n)

    rng = np.random.default_rng(seed)

    candidates = [np.ones(N, dtype=int), sign_pm1_real(psi)]
    for _ in range(max(0, num_random_trials - len(candidates))):
        candidates.append(rng.choice([-1, 1], size=N))

    best_score = -np.inf
    best_b2 = None
    best_flat = None

    for b2 in candidates:
        flat = H @ (b2 * psi)
        score = np.linalg.norm(flat, ord=1)
        if score > best_score:
            best_score = score
            best_b2 = b2.copy()
            best_flat = flat.copy()

    b1 = sign_pm1_real(best_flat)
    uniform = np.ones(N, dtype=float) / np.sqrt(N)
    phi = best_b2 * (H @ (b1 * uniform))
    phi = phi / np.linalg.norm(phi)

    overlap = float(np.dot(psi, phi))

    return {
        "phi": phi.astype(complex),
        "b1": b1,
        "b2": best_b2,
        "overlap": overlap
    }


def residual_decomposition_real(
    target_real: np.ndarray,
    eps: float = 1e-8,
    max_terms: int = 16,
    num_random_trials: int = 64,
    seed: int = 1234
) -> Dict[str, Any]:
    """
    Practical residual refinement inspired by the modern construction.
    """
    psi = np.asarray(target_real, dtype=float).flatten()
    psi = psi / np.linalg.norm(psi)

    residual = psi.copy()
    approx = np.zeros_like(psi, dtype=float)
    terms = []

    for k in range(max_terms):
        rnorm = np.linalg.norm(residual)
        if rnorm <= eps:
            break

        direction = residual / rnorm
        coarse = build_coarse_bhbh_state_real(
            direction,
            num_random_trials=num_random_trials,
            seed=seed + k
        )
        phi_k = np.real(coarse["phi"])
        coeff = float(np.dot(residual, phi_k))

        approx += coeff * phi_k
        residual = psi - approx

        terms.append({
            "coeff": coeff,
            "phi": phi_k.astype(complex),
            "b1": coarse["b1"],
            "b2": coarse["b2"],
            "overlap": coarse["overlap"],
            "residual_norm_after": float(np.linalg.norm(residual)),
        })

    if np.linalg.norm(approx) > 1e-14:
        approx = approx / np.linalg.norm(approx)
    else:
        approx[0] = 1.0

    return {
        "terms": terms,
        "approx_real": approx.astype(complex),
        "final_error": float(np.linalg.norm(psi - approx))
    }


def choose_num_terms(eps: float, min_terms: int = 4) -> int:
    """
    Choose T = 2^t in a practical way.
    """
    if eps <= 0:
        raise ValueError("eps must be positive.")
    t = max(2, int(np.ceil(np.log2(max(2.0, np.log2(1.0 / eps))))))
    return max(2 ** t, min_terms)


def pad_terms_to_power_of_two(
    terms: List[Dict[str, Any]],
    T: int,
    n: int
) -> List[Dict[str, Any]]:
    padded = list(terms)
    N = 2 ** n
    trivial_phi = np.zeros(N, dtype=complex)
    trivial_phi[0] = 1.0
    trivial_signs = np.ones(N, dtype=int)

    while len(padded) < T:
        padded.append({
            "coeff": 0.0,
            "phi": trivial_phi.copy(),
            "b1": trivial_signs.copy(),
            "b2": trivial_signs.copy(),
            "overlap": 1.0,
            "residual_norm_after": 0.0
        })

    return padded[:T]

# Explicit basis-state selective phase synthesis
# Allowed ingredients only:
#   - 1-qubit gates
#   - multi-controlled RZ
#   - ancillas

In [45]:
def append_basis_state_phase(
    qc: QuantumCircuit,
    reg: List,
    bit_pattern: List[int],
    angle: float,
    work_qubit
):
    """
    Apply phase e^{i angle} to exactly one basis state of `reg`.

    Method:
      - X on 0-bits to map desired pattern -> all-ones
      - mcrz(2*angle) on a work ancilla prepared in |1>
      - undo X

    Since RZ(2 angle)|1> = e^{i angle}|1>, the marked basis state
    picks up the desired phase.
    """
    if len(reg) == 0:
        qc.rz(2 * angle, work_qubit)
        return

    if len(reg) != len(bit_pattern):
        raise ValueError("Register length and bit_pattern length differ.")

    # Map desired pattern to all-ones
    for q, b in zip(reg, bit_pattern):
        if b == 0:
            qc.x(q)

    # Multi-controlled RZ on work ancilla
    qc.mcrz(2 * angle, reg, work_qubit)

    # Undo mapping
    for q, b in zip(reg, bit_pattern):
        if b == 0:
            qc.x(q)


def append_boolean_sign_oracle(
    qc: QuantumCircuit,
    reg: List,
    signs_pm1: np.ndarray,
    work_qubit
):
    """
    Implement a Boolean phase oracle:
        |x> -> signs_pm1[x] |x>
    where signs_pm1[x] in {+1, -1}.

    The -1 phases are implemented as angle = pi.
    """
    signs_pm1 = np.asarray(signs_pm1, dtype=int)
    N = len(signs_pm1)
    n = len(reg)

    if N != 2 ** n:
        raise ValueError("sign array length must be 2^n.")

    for x in range(N):
        if signs_pm1[x] == -1:
            bits = int_to_lsb_bits(x, n)
            append_basis_state_phase(qc, reg, bits, np.pi, work_qubit)


def append_controlled_boolean_sign_oracle(
    qc: QuantumCircuit,
    reg_index: List,
    reg_data: List,
    sign_patterns: List[np.ndarray],
    work_qubit
):
    """
    Implement one oracle on |k>|x> with phase sign_patterns[k][x].
    """
    t = len(reg_index)
    n = len(reg_data)
    T = len(sign_patterns)

    for k in range(T):
        k_bits = int_to_lsb_bits(k, t)
        pattern = np.asarray(sign_patterns[k], dtype=int)
        if len(pattern) != 2 ** n:
            raise ValueError("Each sign pattern must have length 2^n.")

        for x in range(2 ** n):
            if pattern[x] == -1:
                x_bits = int_to_lsb_bits(x, n)
                append_basis_state_phase(
                    qc,
                    reg_index + reg_data,
                    k_bits + x_bits,
                    np.pi,
                    work_qubit
                )


def append_general_diagonal_phase_oracle(
    qc: QuantumCircuit,
    reg: List,
    phases: np.ndarray,
    work_qubit,
    tol: float = 1e-14
):
    """
    Implement
        |x> -> exp(i * phases[x]) |x>
    explicitly, basis state by basis state.
    """
    phases = np.asarray(phases, dtype=float)
    n = len(reg)

    if len(phases) != 2 ** n:
        raise ValueError("phases length must be 2^n.")

    for x, theta in enumerate(phases):
        if abs(theta) > tol:
            bits = int_to_lsb_bits(x, n)
            append_basis_state_phase(qc, reg, bits, theta, work_qubit)

# Product-state preparation for geometric LCU weights

In [46]:
def append_geometric_index_preparation(qc: QuantumCircuit, reg_index, beta: float):
    """
    Prepare amplitudes proportional to beta^{k/2} over k in binary.
    """
    t = len(reg_index)
    for j in range(t):
        r = beta ** (2 ** j / 2.0)
        theta = 2.0 * np.arctan(r)
        qc.ry(theta, reg_index[j])


def append_inverse_geometric_index_preparation(qc: QuantumCircuit, reg_index, beta: float):
    t = len(reg_index)
    for j in reversed(range(t)):
        r = beta ** (2 ** j / 2.0)
        theta = 2.0 * np.arctan(r)
        qc.ry(-theta, reg_index[j])


# Reflections

In [47]:
def append_reflection_about_zero(
    qc: QuantumCircuit,
    reg: List,
    work_qubit
):
    """
    Apply reflection 2|0...0><0...0| - I on `reg`, up to a global phase.
    """
    zero_bits = [0] * len(reg)
    append_basis_state_phase(qc, reg, zero_bits, np.pi, work_qubit)


# Build V explicitly using synthesized Boolean phase oracles

In [48]:
def build_V_real_explicit(
    target_real: np.ndarray,
    eps: float = 1e-8,
    max_terms: int = None,
    num_random_trials: int = 64,
    seed: int = 1234
) -> Tuple[QuantumCircuit, Dict[str, Any]]:
    """
    Build the real-amplitude flagged LCU circuit V explicitly.
    """
    target_real = np.asarray(target_real, dtype=float).flatten()
    target_real = target_real / np.linalg.norm(target_real)

    n = num_qubits_from_state(target_real)
    beta = 1.0 / np.sqrt(2.0)

    if max_terms is None:
        T = choose_num_terms(eps)
    else:
        T = 1
        while T < max_terms:
            T *= 2

    t = int(np.log2(T))

    decomp = residual_decomposition_real(
        target_real,
        eps=eps / 4,
        max_terms=T,
        num_random_trials=num_random_trials,
        seed=seed
    )
    terms = pad_terms_to_power_of_two(decomp["terms"], T, n)

    weights = np.array([beta ** k for k in range(T)], dtype=float)

    lcu_vec = np.zeros(2 ** n, dtype=complex)
    for k in range(T):
        lcu_vec += weights[k] * terms[k]["phi"]

    lcu_norm = np.linalg.norm(lcu_vec)
    if lcu_norm < 1e-12:
        raise ValueError("LCU norm is too small. Increase max_terms or random trials.")

    phi_real = lcu_vec / lcu_norm
    xi = np.sqrt((1 - beta) / (1 - beta ** T)) * lcu_norm

    alpha = np.sin(np.pi / 10)
    if xi < alpha - 1e-10:
        raise ValueError(
            f"xi={xi:.6f} is below sin(pi/10)={alpha:.6f}. "
            "Increase max_terms or improve the decomposition."
        )

    ratio = alpha / xi
    ratio = min(max(ratio, 0.0), 1.0)
    theta_g = 2.0 * np.arccos(ratio)

    reg_index = QuantumRegister(t, "a")
    reg_flag = QuantumRegister(1, "f")
    reg_data = QuantumRegister(n, "b")
    reg_work = QuantumRegister(1, "w")   # helper ancilla for explicit phase synthesis

    qc = QuantumCircuit(reg_index, reg_flag, reg_data, reg_work, name="V_real_explicit")

    # work ancilla in |1>
    qc.x(reg_work[0])

    # one-qubit layer on index register
    append_geometric_index_preparation(qc, reg_index, beta)

    # H on data
    qc.h(reg_data)

    # first synthesized oracle
    b1_patterns = [term["b1"] for term in terms]
    append_controlled_boolean_sign_oracle(
        qc, list(reg_index), list(reg_data), b1_patterns, reg_work[0]
    )

    # H on data
    qc.h(reg_data)

    # second synthesized oracle
    b2_patterns = [term["b2"] for term in terms]
    append_controlled_boolean_sign_oracle(
        qc, list(reg_index), list(reg_data), b2_patterns, reg_work[0]
    )

    # inverse one-qubit layer on index register
    append_inverse_geometric_index_preparation(qc, reg_index, beta)

    # extra ancilla tuning so flagged amplitude is sin(pi/10)
    qc.ry(theta_g, reg_flag[0])

    info = {
        "n": n,
        "T": T,
        "t_index": t,
        "beta": beta,
        "terms": terms,
        "phi_real": phi_real,
        "xi_before_flag_tuning": xi,
        "alpha_after_flag_tuning": alpha,
        "decomp_error": decomp["final_error"],
    }

    return qc, info

# Exact amplitude amplification on the logical registers

In [49]:
def build_exact_amplitude_amplified_real_explicit(
    target_real: np.ndarray,
    eps: float = 1e-8,
    max_terms: int = None,
    num_random_trials: int = 64,
    seed: int = 1234
) -> Tuple[QuantumCircuit, Dict[str, Any]]:
    """
    Build the full real-state preparation circuit using
    two rounds of exact amplitude amplification.
    """
    V, info = build_V_real_explicit(
        target_real=target_real,
        eps=eps,
        max_terms=max_terms,
        num_random_trials=num_random_trials,
        seed=seed
    )

    reg_index = V.qregs[0]
    reg_flag = V.qregs[1]
    reg_data = V.qregs[2]
    reg_work = V.qregs[3]

    logical_A = list(reg_index) + list(reg_flag)
    logical_AB = list(reg_index) + list(reg_flag) + list(reg_data)

    qc = QuantumCircuit(*V.qregs, name="AA_real_explicit")
    qc.compose(V, inplace=True)

    def append_R2(circ: QuantumCircuit):
        append_reflection_about_zero(circ, logical_A, reg_work[0])

    def append_R1(circ: QuantumCircuit):
        circ.compose(V.inverse(), inplace=True)
        append_reflection_about_zero(circ, logical_AB, reg_work[0])
        circ.compose(V, inplace=True)

    # Q = R1 R2 ; apply Q twice after V
    append_R2(qc)
    append_R1(qc)
    append_R2(qc)
    append_R1(qc)

    info["num_logical_ancillas"] = len(logical_A)
    info["num_total_ancillas_including_work"] = len(logical_A) + 1

    return qc, info

# Full complex-state version with explicit final phase synthesis

In [56]:
def build_full_paper_style_explicit(
    target: np.ndarray,
    eps: float = 1e-8,
    max_terms: int = None,
    num_random_trials: int = 64,
    seed: int = 1234
) -> Tuple[QuantumCircuit, Dict[str, Any]]:
    """
    Full explicit version:
      1) prepare magnitude state with real-amplitude algorithm
      2) apply final diagonal phase correction explicitly
    """
    target = normalize_state(target)
    n = num_qubits_from_state(target)

    mags = np.abs(target)
    phases = np.angle(target)

    qc_real, info = build_exact_amplitude_amplified_real_explicit(
        target_real=mags,
        eps=eps,
        max_terms=max_terms,
        num_random_trials=num_random_trials,
        seed=seed
    )

    reg_index = qc_real.qregs[0]
    reg_flag = qc_real.qregs[1]
    reg_data = qc_real.qregs[2]
    reg_work = qc_real.qregs[3]

    qc = QuantumCircuit(*qc_real.qregs, name="paper_style_explicit")
    qc.compose(qc_real, inplace=True)

    # Explicit final phase synthesis on the data register only
    append_general_diagonal_phase_oracle(
        qc,
        list(reg_data),
        phases,
        reg_work[0]
    )

    info["target"] = target
    info["magnitude_state"] = mags
    info["phase_angles"] = phases

    return qc, info

# Verification

In [51]:
def ideal_full_state_with_ancillas(target: np.ndarray, qc: QuantumCircuit) -> np.ndarray:
    """
    Ideal final state ordering:
      |0_index>|0_flag>|target_data>|1_work>
    consistent with the register construction used above.
    """
    target = normalize_state(target)

    reg_index = qc.qregs[0]
    reg_flag = qc.qregs[1]
    reg_data = qc.qregs[2]
    reg_work = qc.qregs[3]

    t = len(reg_index)
    f = len(reg_flag)
    n = len(reg_data)
    w = len(reg_work)

    if f != 1 or w != 1:
        raise ValueError("This helper assumes 1 flag ancilla and 1 work ancilla.")

    total_dim = 2 ** (t + f + n + w)
    ideal = np.zeros(total_dim, dtype=complex)

    # register order is little-endian in Qiskit basis indexing
    # basis index = index_bits + flag_bit + data_bits + work_bit
    for x in range(2 ** n):
        bits = [0] * t + [0] + int_to_lsb_bits(x, n) + [1]
        idx = sum(bit << i for i, bit in enumerate(bits))
        ideal[idx] = target[x]

    return ideal


def verify_full_circuit(target: np.ndarray, qc: QuantumCircuit) -> Dict[str, float]:
    actual = Statevector.from_instruction(qc).data
    ideal = ideal_full_state_with_ancillas(target, qc)

    return {
        "fidelity": float(state_fidelity(actual, ideal)),
        "l2_error": float(l2_error(actual, ideal))
    }


def gate_summary(qc: QuantumCircuit) -> Dict[str, int]:
    counts = {}
    for inst, _, _ in qc.data:
        counts[inst.name] = counts.get(inst.name, 0) + 1
    return counts


# Demo for n = 4

In [58]:
if __name__ == "__main__":
    rng = np.random.default_rng(2026)

    n = 4
    N = 2 ** n

    target = rng.normal(size=N) + 1j * rng.normal(size=N)
    target = normalize_state(target)

    qc, info = build_full_paper_style_explicit(
        target,
        eps=1e-6,
        max_terms=8,
        num_random_trials=128,
        seed=2026
    )

    result = verify_full_circuit(target, qc)

    print("\n=== Explicit David Gosset et al-style state preparation (n=4) ===")
    print(f"Total qubits                         : {qc.num_qubits}")
    print(f"Data qubits                          : {n}")
    print(f"Index register size                  : {info['t_index']}")
    print(f"T (number of LCU terms)              : {info['T']}")
    print(f"Residual decomposition error         : {info['decomp_error']:.6e}")
    print(f"xi before flag tuning                : {info['xi_before_flag_tuning']:.6f}")
    print(f"alpha after flag tuning              : {info['alpha_after_flag_tuning']:.6f}")
    print(f"Full-state fidelity                  : {result['fidelity']:.12f}")
    print(f"Full-state L2 error                  : {result['l2_error']:.6e}")

    print("\nGate summary:")
    print(gate_summary(qc))

    print("\nCircuit:")
    print(qc.draw(fold=140))


=== Explicit David Gosset et al-style state preparation (n=4) ===
Total qubits                         : 9
Data qubits                          : 4
Index register size                  : 3
T (number of LCU terms)              : 8
Residual decomposition error         : 8.233423e-04
xi before flag tuning                : 0.796978
alpha after flag tuning              : 0.309017
Full-state fidelity                  : 0.470364913403
Full-state L2 error                  : 7.926770e-01

Gate summary:


/tmp/ipykernel_955/3126369053.py:47: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for inst, _, _ in qc.data:


Streaming output truncated to the last 5000 lines.
«     ┌─┴─┐┌───┐┌─┴─┐┌─────┐┌─┴─┐┌───┐ ┌───┐ ┌──────────┐┌───┐┌────────┐                                                                  »
«  w: ┤ X ├┤ T ├┤ X ├┤ Tdg ├┤ X ├┤ T ├─┤ H ├─┤ Rz(-π/2) ├┤ H ├┤ P(π/8) ├──────────────────────────────────────────────────────────────────»
«     └───┘└───┘└───┘└─────┘└───┘└───┘ └───┘ └──────────┘└───┘└────────┘                                                                  »
«                                                                                                                                        »
«a_0: ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────»
«                                                                                                                                        »
«a_1: ──────────────────────────────────────────────────────────────────────────────────────────────────────────